# Step 2 · Homepage and affiliation from each RePEc profile

**What this notebook does:** for every economist in `repec_people.csv` (made by step 1), it opens their RePEc
profile and copies two things:
- the **homepage** link from the "Personal Details" box,
- their first **affiliation** from the "Affiliation" box just below it. Step 5 uses it for the economists the
  rankings list without an institution: about 350 living economists, often because their time is split between
  several places. Joseph Stiglitz's, for example, is split four ways: three parts of Columbia University, and NBER.

It saves `repec_profiles.csv`: each person's RePEc ID, homepage and affiliation (empty if they don't list one).

**Why it opens one page per person.** The rankings only list names and institutions. The homepage is only on
each person's own profile page, in the "Personal Details" box near the top. For example, on
[Susan Athey's profile](https://ideas.repec.org/e/pat6.html):

| Personal Details | |
|---|---|
| First Name: | Susan |
| Last Name: | Athey |
| Homepage: | https://people.stanford.edu/athey/ |

With about 5,600 economists, that is about 5,600 pages.

**How long it takes: about 15 minutes.** Two things keep it quick while going easy on RePEc, which is run by
volunteers:
- it keeps one connection to RePEc open for all the pages, instead of connecting again for every page,
- it asks for one page at a time, never several at once, with a short pause in between.

**If it stops part-way.** Every result is added to `repec_profiles.csv` as soon as it is read. If the run stops
(or you stop it), just run the notebook again: it skips everyone already in the file and carries on.
To start over, for example to get fresh homepages next year, delete `repec_profiles.csv` first.

## Setup

- `requests` downloads web pages, `BeautifulSoup` reads their HTML, `pandas` handles tables.
- `csv` writes one line at a time to the output file; `os` checks whether a file exists; `time` does the pauses.

In [1]:
import csv
import os
import time
import warnings

# The Mac's built-in Python prints a harmless warning about its SSL library when `requests` loads.
# Hide it. (This line has to come before `import requests`.)
warnings.filterwarnings("ignore")

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Every request says who is asking. Some websites refuse requests that don't.
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; EconNobel poll; https://panahovf.github.io/econnobel/)"}

OUTPUT = "repec_profiles.csv"    # where the results go
PAUSE = 0.1                      # seconds to wait between pages (a tenth of a second)

## 1. Who still needs to be read

The economists come from step 1. If an earlier run was stopped part-way, the people it already read are in
`repec_profiles.csv`, so we skip them. On a first run, we create that file with just its column names.

In [2]:
people = pd.read_csv("repec_people.csv")

if os.path.exists(OUTPUT):
    # An earlier run saved some results: note whose profiles are done.
    already_read = set(pd.read_csv(OUTPUT)["repec_id"])
else:
    # First run: a new file with only the column names in it.
    already_read = set()
    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["repec_id", "homepage", "affiliation"])

# Everyone not read yet. (~ means "not".)
to_read = people[~people["repec_id"].isin(already_read)]

print(len(already_read), "profiles read before;", len(to_read), "to go")

0 profiles read before; 5644 to go


## 2. Read each profile

In the page's HTML, the homepage is the link in the table cell right after the one marked `homelabel`:

```html
<td class="homelabel"></td><td><a href="https://people.stanford.edu/athey/">…</a></td>
```

and the affiliations are in a box marked `affiliation`, one heading per place, written from the department up
to the university, with the share of the person's time in front:

```html
<div id="affiliation"> <h3>(99%) Graduate School of Business<br>Stanford University</h3> …
```

For each person, the loop below:
1. downloads their profile page,
2. checks it really is a profile (it has "Personal Details"); if RePEc sent something else, such as an error
   page, the person is skipped and the next run tries them again,
3. takes the homepage link and the last line of the first affiliation ("Stanford University"),
4. adds a line to `repec_profiles.csv` straight away.

It prints its progress every 500 profiles.

In [3]:
# A Session keeps one connection to RePEc open for all the pages, instead of connecting again for every page.
# That alone makes each page about three times quicker.
session = requests.Session()
session.headers.update(HEADERS)

failures_in_a_row = 0          # pages in a row that didn't come back as a profile
number = 0                     # how many people we have gone through so far
start_time = time.time()

for _, person in to_read.iterrows():
    number = number + 1

    # 1. Download the profile page. If the download itself fails (say, the connection drops), use empty text:
    #    the check just below then skips this person, and the next run tries them again.
    try:
        response = session.get(person["profile_url"], timeout=60)
        response.encoding = "utf-8"          # RePEc pages are UTF-8
        page_text = response.text
    except requests.RequestException:
        page_text = ""
    time.sleep(PAUSE)                        # a short pause before the next page

    # 2. Is it really a profile? If not, skip this person. After 5 in a row, stop: RePEc is probably busy or
    #    asking us to slow down. Everything read so far is saved, so run the notebook again later.
    if "Personal Details" not in page_text:
        failures_in_a_row = failures_in_a_row + 1
        if failures_in_a_row == 5:
            print("Stopped: 5 pages in a row were not profiles. Run this notebook again later; it carries on from here.")
            break
        continue
    failures_in_a_row = 0

    # 3. Read the top of the page. The Personal Details and the affiliations are near the top (about 13,000
    #    characters in), while the rest of the page lists all their publications. So we only read the first
    #    30,000 characters, which is much quicker than reading the whole page.
    page = BeautifulSoup(page_text[:30000], "html.parser")

    # 3a. The homepage: the link in the table cell right after the one marked "homelabel".
    homepage = ""
    label_cell = page.find("td", class_="homelabel")
    if label_cell is not None and label_cell.find_next_sibling("td") is not None:
        link = label_cell.find_next_sibling("td").find("a", href=True)
        if link is not None:
            homepage = link["href"].strip()

    # 3b. The affiliation: the first heading in the "affiliation" box, e.g.
    #     "(25%) Department of Economics / School of Arts and Sciences / Columbia University".
    #     Its lines go from the department up to the university, so the last line is the institution.
    affiliation = ""
    affiliation_box = page.find("div", id="affiliation")
    if affiliation_box is not None and affiliation_box.find("h3") is not None:
        # .stripped_strings gives the heading's lines as a list: ["(25%) Department of Economics", ..., "Columbia University"]
        lines = list(affiliation_box.find("h3").stripped_strings)
        affiliation = lines[-1]
        # A place written on one line still starts with the share, "(25%) National Bureau of ...": remove it.
        if affiliation.startswith("(") and "%) " in affiliation:
            affiliation = affiliation.split("%) ", 1)[1]

    # 4. Add one line to the output file right away ("a" means append), so nothing is lost if the run stops.
    with open(OUTPUT, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([person["repec_id"], homepage, affiliation])

    if number % 500 == 0:
        minutes = (time.time() - start_time) / 60
        print(number, "of", len(to_read), "profiles read, after", round(minutes, 1), "minutes")

print("Done:", number, "profiles gone through in", round((time.time() - start_time) / 60, 1), "minutes")

500 of 5644 profiles read, after 1.3 minutes


1000 of 5644 profiles read, after 2.7 minutes


1500 of 5644 profiles read, after 4.0 minutes


2000 of 5644 profiles read, after 5.3 minutes


2500 of 5644 profiles read, after 6.6 minutes


...

4000 of 5644 profiles read, after 10.6 minutes


4500 of 5644 profiles read, after 12.0 minutes


5000 of 5644 profiles read, after 13.3 minutes


5500 of 5644 profiles read, after 14.7 minutes


Done: 5644 profiles gone through in 15.1 minutes


## 3. Check the result

How many economists list a homepage and an affiliation, and whether anyone is still missing.

In [4]:
profiles = pd.read_csv(OUTPUT).fillna("")    # fillna(""): empty cells become empty text

print(len(profiles), "of", len(people), "profiles read")
print((profiles["homepage"] != "").sum(), "list a homepage")
print((profiles["affiliation"] != "").sum(), "list an affiliation")

missing = (~people["repec_id"].isin(profiles["repec_id"])).sum()
if missing > 0:
    print(missing, "profiles could not be read yet: run this notebook again to try them again")

5644 of 5644 profiles read
4344 list a homepage
5281 list an affiliation


A few examples, with the names and institutions from step 1. The `institution` column (from the rankings)
is empty for some, and `affiliation` (from the profile) fills the gap.

In [5]:
examples = people.merge(profiles, on="repec_id")
examples[["name", "institution", "affiliation", "homepage"]].head(10)

,name,institution,affiliation,homepage
0,Andrei Shleifer,Harvard University,Harvard University,https://scholar.harvard.edu/shleifer/home
1,Daron Acemoglu,Massachusetts Institute of Technology (MIT),Massachusetts Institute of Technology (MIT),https://economics.mit.edu/people/faculty/daron...
2,James J. Heckman,University of Chicago,University of Chicago,http://heckman.uchicago.edu
3,Joseph Stiglitz,NaN,Columbia University,http://www.gsb.columbia.edu/faculty/jstiglitz/
4,Jean Tirole,Toulouse School of Economics (TSE),Toulouse School of Economics (TSE),https://www.tse-fr.eu/people/jean-tirole
5,Robert J. Barro,Harvard University,Harvard University,https://scholar.harvard.edu/barro
6,John List,University of Chicago,National Bureau of Economic Research (NBER),https://voices.uchicago.edu/jlist/
7,Peter C. B. Phillips,NaN,Yale University,http://korora.econ.yale.edu
8,"Eugene F. Fama, Sr.",University of Chicago,University of Chicago,
9,David E. Card,NaN,University of California-Berkeley,
